In [1]:
# ── 셀 1 │ 경로 & 라이브러리 ──
import os, re, json, time, datetime
import numpy as np
import pandas as pd

NEW      = 'C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/'
CM_XLSX  = NEW + 'data/raw/crediblemeds/QT_drug_lists.xlsx'
CM_OUT   = NEW + 'data/processed/crediblemeds/'
CLIN_POOL= NEW + 'data/processed/clinical/clinical_pool.csv'
HERG_POOL= NEW + 'data/processed/herg_karim/herg_karim_pool.csv'
os.makedirs(CM_OUT, exist_ok=True)

for p in [CM_XLSX, CLIN_POOL, HERG_POOL]:
    print(('OK      ' if os.path.exists(p) else 'MISSING '), p)

xl = pd.ExcelFile(CM_XLSX)
print('\n시트:', xl.sheet_names)

OK       C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/data/raw/crediblemeds/QT_drug_lists.xlsx
OK       C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/data/processed/clinical/clinical_pool.csv
OK       C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/data/processed/herg_karim/herg_karim_pool.csv

시트: ['Sheet1', 'Sheet2']


In [2]:
# ── 셀 2 │ 시트 구조 정찰 (출력 보내주세요) ──
for sh in xl.sheet_names:
    raw = pd.read_excel(CM_XLSX, sheet_name=sh, header=None, nrows=12)
    print('=' * 70)
    print(f'[{sh}]  shape(앞 12행) = {raw.shape}')
    print(raw.to_string(max_colwidth=40))

[Sheet1]  shape(앞 12행) = (12, 9)
                             0                                        1                           2                                        3                        4            5          6                    7                            8
0                 Generic Name               Brand Names (Partial List)                  Drug Class                          Therapeutic Use            Risk Category  QT Increase  TdP Cases  ECG Recommendations  Warning for Congenital LQTS
1                     Abarelix                                 Plenaxis             GnRH antagonist                        Cancer (prostate)     Possible Risk of TdP          Yes         No                   No                      Caution
2                  Abiraterone     Zytiga, Abiratas, Abretone, Abirapro               Anti-androgen                        Cancer (Prostate)  Conditional Risk of TdP          Yes        Yes                  Yes                           No
3      

In [7]:
# ── 셀 3 │ 파싱 (교체본) ──
# Sheet1: 정상 헤더. Sheet2: 헤더 없음 + 컬럼 순서 다름(link가 5번째).
S2_COLS = ['Generic Name', 'Brand Names (Partial List)', 'Drug Class',
           'Therapeutic Use', 'link', 'Risk Category']

df1 = pd.read_excel(CM_XLSX, sheet_name='Sheet1', header=0)
df1.columns = [str(c).strip() for c in df1.columns]
df1['sheet'] = 'Sheet1'

df2 = pd.read_excel(CM_XLSX, sheet_name='Sheet2', header=None)
df2 = df2.iloc[:, :len(S2_COLS)]
df2.columns = S2_COLS
df2['sheet'] = 'Sheet2'

cm = pd.concat([df1, df2], ignore_index=True, sort=False)

# 'Risk Category' 가 비어있는 행은 주석 행 ('(Only on Non US Market)' 등)
junk = cm['Risk Category'].isna().sum()
cm = cm[cm['Risk Category'].notna()].copy()
print(f'주석/빈 행 {junk}개 제거 → {len(cm)}행')

TIER = {'known': 'Known', 'possible': 'Possible', 'conditional': 'Conditional'}
cm['tier'] = cm['Risk Category'].str.lower().map(
    lambda s: next((v for k, v in TIER.items() if k in str(s)), None))
print('\n등급 매핑 실패:', cm.loc[cm['tier'].isna(), 'Risk Category'].unique())
cm = cm[cm['tier'].notna()].copy()

# 선천성 LQTS 경고는 별도 플래그 (네 번째 리스트에 해당)
if 'Warning for Congenital LQTS' in cm.columns:
    cm['clqts'] = (cm['Warning for Congenital LQTS'].astype(str)
                   .str.strip().str.lower()
                   .isin(['avoid', 'caution']).astype(int))
else:
    cm['clqts'] = 0

print('\n', cm['tier'].value_counts())
print('\nSheet 별:', cm.groupby(['sheet', 'tier']).size().to_dict())
print('선천성 LQTS 경고:', int(cm['clqts'].sum()))

주석/빈 행 3개 제거 → 294행

등급 매핑 실패: []

 tier
Possible       168
Known           70
Conditional     56
Name: count, dtype: int64

Sheet 별: {('Sheet1', 'Conditional'): 54, ('Sheet1', 'Known'): 69, ('Sheet1', 'Possible'): 159, ('Sheet2', 'Conditional'): 2, ('Sheet2', 'Known'): 1, ('Sheet2', 'Possible'): 9}
선천성 LQTS 경고: 92


In [9]:
# ── 셀 4 │ 약물명 정규화 (교체본) ──
NAME_COL = 'Generic Name'

SALT = (r'\b(hydrochloride|hcl|hydrobromide|mesylate|mesilate|maleate|tartrate|'
        r'citrate|sulfate|sulphate|phosphate|succinate|fumarate|acetate|'
        r'sodium|potassium|calcium|besylate|tosylate|lactate|pamoate|'
        r'monohydrate|dihydrate|anhydrous)\b')

def norm_name(s):
    if pd.isna(s):
        return None
    s = str(s).lower().strip()
    s = re.sub(r'[\*†‡#]', ' ', s)
    s = re.sub(SALT, ' ', s)
    s = re.sub(r'[^a-z0-9 \-]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s or None

def name_variants(s):
    """'Alimemazine (Trimeprazine)' -> ['alimemazine', 'trimeprazine']
       괄호 안은 동의어이므로 버리지 않고 대체 후보로 쓴다."""
    if pd.isna(s):
        return []
    s = str(s)
    inside = re.findall(r'\((.*?)\)', s)
    outside = re.sub(r'\(.*?\)', ' ', s)
    cands = [outside] + inside
    out = []
    for c in cands:
        for part in re.split(r'\s*[/;]\s*', c):
            n = norm_name(part)
            if n and len(n) > 2 and n not in out:
                out.append(n)
    return out

cm['variants'] = cm[NAME_COL].map(name_variants)
cm['name_primary'] = cm['variants'].map(lambda v: v[0] if v else None)
cm = cm[cm['name_primary'].notna()].copy()

# 같은 약이 두 시트에 모두 있으면 강한 등급을 남긴다
RANK = {'Known': 3, 'Possible': 2, 'Conditional': 1}
cm['rank'] = cm['tier'].map(RANK)
dup = cm['name_primary'].duplicated().sum()
cm_u = (cm.sort_values('rank', ascending=False)
          .drop_duplicates('name_primary', keep='first')
          .reset_index(drop=True))

print(f'원본 {len(cm)}행 → 고유 약물 {len(cm_u)}개 (시트 간 중복 {dup}건)')
print(cm_u['tier'].value_counts())
print('\n동의어가 잡힌 예:')
print(cm_u.loc[cm_u['variants'].map(len) > 1, [NAME_COL, 'variants']].head(10).to_string())

원본 300행 → 고유 약물 282개 (시트 간 중복 18건)
tier
Possible       159
Known           69
Conditional     54
Name: count, dtype: int64

동의어가 잡힌 예:
                            Generic Name                              variants
33   Levomepromazine (Methotrimeprazine)  [levomepromazine, methotrimeprazine]
44       Papaverine HCl (Intra-coronary)          [papaverine, intra-coronary]
70        Zuclopenthixol (Zuclopentixol)       [zuclopenthixol, zuclopentixol]
109               Tipiracil/Trifluridine             [tipiracil, trifluridine]
112          Saquinavir (with ritonavir)          [saquinavir, with ritonavir]
131             Pitolisant (Tiprolisant)             [pitolisant, tiprolisant]
148                  Lopinavir/Ritonavir                [lopinavir, ritonavir]
160           Dextromethorphan/Quinidine         [dextromethorphan, quinidine]
179           Alimemazine (Trimeprazine)           [alimemazine, trimeprazine]
183              Artemether/Lumefantrine            [artemether, lumefantri

In [12]:
# ── 셀 5 │ 이름 → InChIKey14 매핑 ──
# 1단계: 임상 풀에 약물명 컬럼이 있으면 그걸로 먼저 매칭 (네트워크 불필요)
clin = pd.read_csv(CLIN_POOL)
print('임상 풀 컬럼:', list(clin.columns))

name_cols = [c for c in clin.columns if re.search(r'name|drug|compound|title', c, re.I)]
print('→ 이름 후보 컬럼:', name_cols)

local = {}
for c in name_cols:
    for nm, k in zip(clin[c].map(norm_name), clin['key14']):
        if nm and isinstance(k, str):
            local.setdefault(nm, k)
print(f'임상 풀 로컬 사전 {len(local)}개')

cm_u['key14'] = cm_u['variants'].map(
    lambda v: next((local[n] for n in v if n in local), None))
print(f'로컬 매칭 {cm_u["key14"].notna().sum()} / {len(cm_u)}')

임상 풀 컬럼: ['canon_smiles', 'key14', 'y', 'n_atoms', 'scaffold', 'orig_split', 'label_qt', 'label_tdp', 'label_qt_tdp', 'in_crediblemeds']
→ 이름 후보 컬럼: []
임상 풀 로컬 사전 0개
로컬 매칭 0 / 282


In [13]:
# ── 셀 6 교체 부분 ──
def resolve(variants):
    for nm in variants:
        r = pubchem_key(nm)
        if r.get('inchikey'):
            return nm, r
    return (variants[0] if variants else None), {'smiles': None, 'inchikey': None}

need = cm_u['key14'].isna()
print(f'PubChem 조회 대상 {need.sum()}개')
res = []
for i, v in enumerate(cm_u.loc[need, 'variants'], 1):
    res.append(resolve(v))
    if i % 25 == 0:
        print(f'  {i}/{need.sum()}')
        json.dump(cache, open(CACHE, 'w', encoding='utf-8'))
json.dump(cache, open(CACHE, 'w', encoding='utf-8'))

cm_u.loc[need, 'matched_name']   = [a for a, _ in res]
cm_u.loc[need, 'pubchem_smiles'] = [b.get('smiles') for _, b in res]
cm_u.loc[need, 'pubchem_ik']     = [b.get('inchikey') for _, b in res]
cm_u['key14'] = cm_u['key14'].fillna(
    cm_u['pubchem_ik'].map(lambda k: k.split('-')[0] if isinstance(k, str) else None))

ok = cm_u['key14'].notna().sum()
print(f'\n구조 해석 {ok} / {len(cm_u)}  (미해석 {len(cm_u)-ok})')
print('미해석:', cm_u.loc[cm_u['key14'].isna(), 'name_primary'].tolist())

PubChem 조회 대상 282개
  25/282
  50/282
  75/282
  100/282
  125/282
  150/282
  175/282
  200/282
  225/282
  250/282
  275/282

구조 해석 277 / 282  (미해석 5)
미해석: ['necitumumab', 'nusinersen', 'tebentafusp', 'hydrocodone - er', 'inotuzumab ozogamicin']


In [17]:
# ── 공용 헬퍼 (각 노트북 첫 셀에 붙여두세요) ──
def pick(df, *cands, required=True):
    """후보 중 df에 실제로 있는 첫 컬럼명을 반환."""
    for c in cands:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f'{cands} 중 어느 것도 없음. 실제 컬럼: {list(df.columns)}')
    return None

YC = lambda df: pick(df, 'y', 'Y', 'label', 'Label', 'activity')
KC = lambda df: pick(df, 'key14', 'inchikey14', 'InChIKey14')

In [18]:
# ── 셀 7 │ 중첩 분석 (교체본) ──
cm_ok = cm_u[cm_u['key14'].notna()].drop_duplicates('key14').copy()
print(f'고유 key14 {len(cm_ok)}개')

clin = pd.read_csv(CLIN_POOL)
ck, cy = KC(clin), YC(clin)
print(f'임상 풀: key={ck} label={cy}')

clin_keys = set(clin[ck].dropna())
cm_in_clin = cm_ok[cm_ok['key14'].isin(clin_keys)]
print(f'\nCredibleMeds ∩ 임상 = {len(cm_in_clin)}   (원고 ChEMBL 기준 125종)')
print(cm_in_clin['tier'].value_counts())

m = clin.merge(cm_ok[['key14', 'tier']].rename(columns={'key14': ck}),
               on=ck, how='left')
m['in_cm'] = m['tier'].notna().astype(int)
print('\n[임상 라벨 × CredibleMeds]')
print(pd.crosstab(m['in_cm'], m[cy], margins=True))

print('\n[등급별 심독성 양성률] — 단조 증가해야 설득력이 있습니다')
print(m[m['in_cm'] == 1].groupby('tier')[cy].agg(['size', 'mean']).round(3))

if os.path.exists(HERG_POOL):
    herg = pd.read_csv(HERG_POOL)
    hk_col, hy_col = KC(herg), YC(herg)
    print(f'\nhERG 풀: key={hk_col} label={hy_col}')
    hpos = set(herg.loc[herg[hy_col] == 1, hk_col].dropna())
    allk = set(herg[hk_col].dropna())
    sub = cm_ok[cm_ok['key14'].isin(allk)].copy()
    sub['hpos'] = sub['key14'].isin(hpos)
    print(f'CredibleMeds ∩ hERG_Karim = {len(sub)}, '
          f'그중 hERG 양성 {sub["hpos"].mean():.1%}')
    print(sub.groupby('tier')['hpos'].agg(['size', 'mean']).round(3))
else:
    print('\nhERG 풀 없음 — 건너뜀')

고유 key14 270개
임상 풀: key=key14 label=y

CredibleMeds ∩ 임상 = 143   (원고 ChEMBL 기준 125종)
tier
Possible       71
Known          39
Conditional    33
Name: count, dtype: int64

[임상 라벨 × CredibleMeds]
y        0    1   All
in_cm                
0      490  592  1082
1       10  133   143
All    500  725  1225

[등급별 심독성 양성률] — 단조 증가해야 설득력이 있습니다
             size   mean
tier                    
Conditional    33  0.879
Known          39  0.923
Possible       71  0.958

hERG 풀: key=key14 label=Y
CredibleMeds ∩ hERG_Karim = 125, 그중 hERG 양성 52.8%
             size   mean
tier                    
Conditional    26  0.538
Known          42  0.667
Possible       57  0.421


In [19]:
# ── 셀 8 │ 저장 ──
cm_ok.to_csv(CM_OUT + 'crediblemeds_pool.csv', index=False, encoding='utf-8-sig')
cm_u.to_csv(CM_OUT + 'crediblemeds_all_names.csv', index=False, encoding='utf-8-sig')

import hashlib
meta = {
    'source': 'CredibleMeds QTdrugs lists (QT_drug_lists.xlsx)',
    'source_md5': hashlib.md5(open(CM_XLSX, 'rb').read()).hexdigest(),
    'sheets': xl.sheet_names,
    'name_column': NAME_COL,
    'n_rows_raw': int(len(cm)),
    'n_unique_names': int(cm_u['name_norm'].nunique()),
    'n_resolved_key14': int(len(cm_ok)),
    'n_unresolved': int(cm_u['key14'].isna().sum()),
    'unresolved_names': cm_u.loc[cm_u['key14'].isna(), 'name_norm'].tolist(),
    'tier_counts': cm_ok['tier'].value_counts().to_dict(),
    'overlap_clinical': int(len(cm_in_clin)),
    'resolver': 'local clinical-pool name match, then PubChem PUG-REST name lookup',
    'salt_stripping': 'common salt/hydrate suffixes removed before lookup',
    'processed_date': datetime.date.today().isoformat(),
}
json.dump(meta, open(CM_OUT + 'crediblemeds_provenance.json', 'w', encoding='utf-8'),
          indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in meta.items() if k != 'unresolved_names'},
                 indent=2, ensure_ascii=False))

{
  "source": "CredibleMeds QTdrugs lists (QT_drug_lists.xlsx)",
  "source_md5": "bc05dacc4f6d8c729a5fdee153560cd2",
  "sheets": [
    "Sheet1",
    "Sheet2"
  ],
  "name_column": "Generic Name",
  "n_rows_raw": 300,
  "n_unique_names": 282,
  "n_resolved_key14": 270,
  "n_unresolved": 5,
  "tier_counts": {
    "Possible": 152,
    "Known": 67,
    "Conditional": 51
  },
  "overlap_clinical": 143,
  "resolver": "local clinical-pool name match, then PubChem PUG-REST name lookup",
  "salt_stripping": "common salt/hydrate suffixes removed before lookup",
  "processed_date": "2026-11-12"
}


In [24]:
print('cm_ok 중복 컬럼:', cm_ok.columns[cm_ok.columns.duplicated()].tolist())
print('cm    중복 컬럼:', cm.columns[cm.columns.duplicated()].tolist())
print('key14 타입:', type(cm_ok['key14']))

cm_ok 중복 컬럼: []
cm    중복 컬럼: []
key14 타입: <class 'pandas.core.series.Series'>


In [26]:
# ── 셀 9 │ 위험 등급 × QT/TdP 교차표 (unique 기준, 완성본) ──
# 중복 컬럼 정리 (앞의 것만 유지) — unhashable Series 오류의 원인
cm    = cm.loc[:,    ~cm.columns.duplicated()].copy()
cm_ok = cm_ok.loc[:, ~cm_ok.columns.duplicated()].copy()
print('정리 후 cm 컬럼:', list(cm.columns))

QTCOL  = next(c for c in cm.columns if re.search(r'qt.*increase', str(c), re.I))
TDPCOL = next(c for c in cm.columns if re.search(r'tdp.*case',    str(c), re.I))
print('컬럼:', QTCOL, '/', TDPCOL)
print('값 분포:', cm[QTCOL].value_counts(dropna=False).to_dict())

yes = lambda s: s.astype(str).str.strip().str.lower().eq('yes')
cm['_qt']  = yes(cm[QTCOL])
cm['_tdp'] = yes(cm[TDPCOL])
cm['_ann'] = cm[QTCOL].notna() | cm[TDPCOL].notna()

ROWS = ['Known', 'Conditional', 'Possible']
RANK = {'Known': 3, 'Possible': 2, 'Conditional': 1}

def xtab(df, label):
    t = pd.DataFrame({
        'Total':      df.groupby('tier').size(),
        'QT,TdP (a)': df.groupby('tier')['_ann'].sum(),
        'QT (b)':     df.groupby('tier')['_qt'].sum(),
        'TdP (c)':    df.groupby('tier')['_tdp'].sum(),
    }).reindex(ROWS).fillna(0).astype(int)
    t.loc['Total'] = t.sum()
    print(f'\n[{label}]')
    print(t.to_string())
    return t

def collapse(df, by):
    """by 단위로 접는다. 등급은 가장 강한 것, 플래그는 any."""
    d = df[df[by].notna()].copy()
    d['_rank'] = d['tier'].map(RANK)
    flags = d.groupby(by)[['_qt', '_tdp', '_ann']].any()
    tier  = (d.sort_values('_rank', ascending=False)
               .drop_duplicates(by, keep='first')
               .set_index(by)['tier'])
    return flags.join(tier).reset_index()

# key14 매핑 — Series 로 확실히 만든 뒤 매핑
k = cm_ok[['name_primary', 'key14']].dropna().drop_duplicates('name_primary')
name2key = dict(zip(k['name_primary'], k['key14']))
cm['key14'] = cm['name_primary'].map(name2key)
print('\nkey14 매핑됨:', int(cm['key14'].notna().sum()), '/', len(cm))
assert cm['key14'].dropna().map(lambda v: isinstance(v, str)).all(), 'key14에 비문자열'

# 1) 원본 행 기준
t_raw = xtab(cm, f'원본 행 기준  n={len(cm)}')

# 2) 약물명 기준
by_name = collapse(cm, 'name_primary')
t_name = xtab(by_name, f'약물명 unique  n={len(by_name)}')

# 3) key14(구조) 기준 ── 원고 표는 이것
by_key = collapse(cm, 'key14')
t_key = xtab(by_key, f'key14 unique  n={len(by_key)}'
                     f'  (구조 미해석 {int(cm["key14"].isna().sum())}행 제외)')

print('\n[참고] key14 기준 합집합·교집합')
ex = pd.DataFrame({
    'QT or TdP':  by_key.groupby('tier')[['_qt', '_tdp']].apply(
                      lambda d: int((d['_qt'] | d['_tdp']).sum())),
    'QT and TdP': by_key.groupby('tier')[['_qt', '_tdp']].apply(
                      lambda d: int((d['_qt'] & d['_tdp']).sum())),
}).reindex(ROWS).fillna(0).astype(int)
ex.loc['Total'] = ex.sum()
print(ex.to_string())

t_key.to_csv(CM_OUT + 'table_crediblemeds_unique.csv', encoding='utf-8-sig')
print('\n[탭 구분 복사용 — key14 기준]')
print('\t' + '\t'.join(t_key.columns))
for i, r in t_key.iterrows():
    print(str(i) + '\t' + '\t'.join(str(v) for v in r.values))

정리 후 cm 컬럼: ['Generic Name', 'Brand Names (Partial List)', 'Drug Class', 'Therapeutic Use', 'Risk Category', 'QT Increase', 'TdP Cases', 'ECG Recommendations', 'Warning for Congenital LQTS', 'sheet', 'link', 'tier', 'clqts', 'name_norm', 'rank', 'variants', 'name_primary', '_qt', '_tdp', '_ann']
컬럼: QT Increase / TdP Cases
값 분포: {'Yes': 167, nan: 71, 'No': 62}

key14 매핑됨: 288 / 300

[원본 행 기준  n=300]
             Total  QT,TdP (a)  QT (b)  TdP (c)
tier                                           
Known           70          43      37       35
Conditional     57          49      24       17
Possible       173         137     106       18
Total          300         229     167       70

[약물명 unique  n=282]
             Total  QT,TdP (a)  QT (b)  TdP (c)
tier                                           
Known           69          43      37       35
Conditional     54          48      24       17
Possible       159         133     103       17
Total          282         224     164       69


In [27]:
# ── 셀 10 │ 임상 풀과 겹치는 약물만 (교체본) ──
inclin = cm[cm['key14'].isin(clin_keys)].copy()
by_key_clin = collapse(inclin, 'key14')
t_clin = xtab(by_key_clin,
              f'임상 풀과 겹치는 CredibleMeds  n={len(by_key_clin)}')

t_clin.to_csv(CM_OUT + 'table_crediblemeds_inclin.csv', encoding='utf-8-sig')
print('\n[탭 구분 복사용 — 임상 중첩]')
print('\t' + '\t'.join(t_clin.columns))
for i, r in t_clin.iterrows():
    print(str(i) + '\t' + '\t'.join(str(v) for v in r.values))


[임상 풀과 겹치는 CredibleMeds  n=143]
             Total  QT,TdP (a)  QT (b)  TdP (c)
tier                                           
Known           39          31      26       25
Conditional     33          30      16       12
Possible        71          69      51       11
Total          143         130      93       48

[탭 구분 복사용 — 임상 중첩]
	Total	QT,TdP (a)	QT (b)	TdP (c)
Known	39	31	26	25
Conditional	33	30	16	12
Possible	71	69	51	11
Total	143	130	93	48
